# Thuật toán Apriori — Association Rule Mining

Dùng thư viện `mlxtend`.

**Các khái niệm:**
- `Support` $s(I)$: tỉ lệ giao dịch có chứa itemset $I$
- `Frequent itemset`: itemset có $s(I) \geq minsup$
- **Apriori principle**: $\forall X, Y: X \subseteq Y \Rightarrow s(X) \geq s(Y)$ (tính chất *anti-monotone*) — nếu 1 itemset không frequent thì mọi tập cha của nó cũng không frequent, nhờ vậy cắt bớt được không gian tìm kiếm.

In [1]:
import numpy as np
import pandas as pd
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

print("mlxtend OK")

mlxtend OK


## 1. Dữ liệu giao dịch

In [2]:
transactions = {
    0: ["Bread", "Milk"],
    1: ["Bread", "Diaper", "Beer", "Eggs"],
    2: ["Milk", "Diaper", "Beer", "Coke"],
    3: ["Bread", "Milk", "Diaper", "Coke"],
    4: ["Bread", "Milk", "Diaper", "Coke"],
}

ts = list(transactions.values())
for tid, items in transactions.items():
    print(tid, items)

0 ['Bread', 'Milk']
1 ['Bread', 'Diaper', 'Beer', 'Eggs']
2 ['Milk', 'Diaper', 'Beer', 'Coke']
3 ['Bread', 'Milk', 'Diaper', 'Coke']
4 ['Bread', 'Milk', 'Diaper', 'Coke']


## 2. Mã hoá one-hot

`TransactionEncoder` biến danh sách giao dịch thành ma trận boolean: mỗi dòng là 1 giao dịch (TID), mỗi cột là 1 item.

In [3]:
te = TransactionEncoder()
te_ary = te.fit_transform(ts)

df = pd.DataFrame(te_ary, columns=te.columns_, index=list(transactions.keys()))
df.index.name = "TID"
df

,Beer,Bread,Coke,Diaper,Eggs,Milk
TID,,,,,,
0,False,True,False,False,False,True
1,True,True,False,True,True,False
2,True,False,True,True,False,True
3,False,True,True,True,False,True
4,False,True,True,True,False,True


## 3. Tìm frequent itemsets

$$
s(I) = \frac{\text{số giao dịch chứa } I}{\text{tổng số giao dịch}}
$$

Chọn `min_support = 0.4`, tức itemset phải xuất hiện ít nhất $0.4 \times 5 = 2$ lần.

In [4]:
MIN_SUPPORT = 0.4

freq_itemsets = apriori(df, min_support=MIN_SUPPORT, use_colnames=True)

freq_itemsets["length"] = freq_itemsets["itemsets"].apply(len)      # k = số item
freq_itemsets["count"]  = (freq_itemsets["support"] * len(df)).round().astype(int)

freq_itemsets = (freq_itemsets
                 .sort_values(["length", "support"], ascending=[True, False])
                 .reset_index(drop=True))
freq_itemsets

,support,itemsets,length,count
0,0.8,frozenset({Bread}),1,4
1,0.8,frozenset({Diaper}),1,4
2,0.8,frozenset({Milk}),1,4
3,0.6,frozenset({Coke}),1,3
4,0.4,frozenset({Beer}),1,2
5,0.6,"frozenset({Bread, Diaper})",2,3
6,0.6,"frozenset({Bread, Milk})",2,3
7,0.6,"frozenset({Diaper, Coke})",2,3
8,0.6,"frozenset({Milk, Coke})",2,3
9,0.6,"frozenset({Milk, Diaper})",2,3


In [5]:
# In theo từng mức F_k giống bảng trong lý thuyết
for k, group in freq_itemsets.groupby("length"):
    print(f"--- F_{k} ---")
    for _, row in group.iterrows():
        items = ", ".join(sorted(row["itemsets"]))
        print(f"  {{{items}}}".ljust(45) + f"support = {row['support']:.2f}  (count = {row['count']})")
    print()

--- F_1 ---
  {Bread}                                    support = 0.80  (count = 4)
  {Diaper}                                   support = 0.80  (count = 4)
  {Milk}                                     support = 0.80  (count = 4)
  {Coke}                                     support = 0.60  (count = 3)
  {Beer}                                     support = 0.40  (count = 2)

--- F_2 ---
  {Bread, Diaper}                            support = 0.60  (count = 3)
  {Bread, Milk}                              support = 0.60  (count = 3)
  {Coke, Diaper}                             support = 0.60  (count = 3)
  {Coke, Milk}                               support = 0.60  (count = 3)
  {Diaper, Milk}                             support = 0.60  (count = 3)
  {Beer, Diaper}                             support = 0.40  (count = 2)
  {Bread, Coke}                              support = 0.40  (count = 2)

--- F_3 ---
  {Coke, Diaper, Milk}                       support = 0.60  (count = 3)
  {Bread, Cok

## 4. Sinh luật kết hợp

Với luật $X \Rightarrow Y$ (trong đó $X \cap Y = \emptyset$):

$$
\text{confidence}(X \Rightarrow Y) = \frac{s(X \cup Y)}{s(X)}
\qquad
\text{lift}(X \Rightarrow Y) = \frac{\text{confidence}(X \Rightarrow Y)}{s(Y)}
$$

- `lift > 1`: X, Y xuất hiện cùng nhau nhiều hơn mức ngẫu nhiên → luật có ý nghĩa
- `lift = 1`: độc lập
- `lift < 1`: loại trừ nhau

In [6]:
MIN_CONFIDENCE = 0.6

rules = association_rules(freq_itemsets, metric="confidence", min_threshold=MIN_CONFIDENCE)

cols = ["antecedents", "consequents", "support", "confidence", "lift", "leverage", "conviction"]
rules = rules[cols].sort_values(["confidence", "lift"], ascending=False).reset_index(drop=True)

print("Tổng số luật:", len(rules))
rules

Tổng số luật: 40


,antecedents,consequents,support,confidence,lift,leverage,conviction
0,"frozenset({Milk, Diaper})",frozenset({Coke}),0.6,1.000000,1.666667,0.24,inf
1,frozenset({Coke}),"frozenset({Milk, Diaper})",0.6,1.000000,1.666667,0.24,inf
2,"frozenset({Bread, Diaper, Milk})",frozenset({Coke}),0.4,1.000000,1.666667,0.16,inf
3,"frozenset({Bread, Coke})","frozenset({Milk, Diaper})",0.4,1.000000,1.666667,0.16,inf
4,frozenset({Coke}),frozenset({Diaper}),0.6,1.000000,1.250000,0.12,inf
5,frozenset({Coke}),frozenset({Milk}),0.6,1.000000,1.250000,0.12,inf
6,frozenset({Beer}),frozenset({Diaper}),0.4,1.000000,1.250000,0.08,inf
7,"frozenset({Milk, Coke})",frozenset({Diaper}),0.6,1.000000,1.250000,0.12,inf
8,"frozenset({Diaper, Coke})",frozenset({Milk}),0.6,1.000000,1.250000,0.12,inf
9,"frozenset({Bread, Coke})",frozenset({Diaper}),0.4,1.000000,1.250000,0.08,inf


In [7]:
# Chỉ giữ các luật thật sự mạnh: lift > 1
strong = rules[rules["lift"] > 1].copy()
strong["rule"] = strong.apply(
    lambda r: "{" + ", ".join(sorted(r["antecedents"])) + "} => {" + ", ".join(sorted(r["consequents"])) + "}",
    axis=1,
)

strong[["rule", "support", "confidence", "lift"]].reset_index(drop=True)

,rule,support,confidence,lift
0,"{Diaper, Milk} => {Coke}",0.6,1.000000,1.666667
1,"{Coke} => {Diaper, Milk}",0.6,1.000000,1.666667
2,"{Bread, Diaper, Milk} => {Coke}",0.4,1.000000,1.666667
3,"{Bread, Coke} => {Diaper, Milk}",0.4,1.000000,1.666667
4,{Coke} => {Diaper},0.6,1.000000,1.250000
5,{Coke} => {Milk},0.6,1.000000,1.250000
6,{Beer} => {Diaper},0.4,1.000000,1.250000
7,"{Coke, Milk} => {Diaper}",0.6,1.000000,1.250000
8,"{Coke, Diaper} => {Milk}",0.6,1.000000,1.250000
9,"{Bread, Coke} => {Diaper}",0.4,1.000000,1.250000


## 5. Ảnh hưởng của ngưỡng

Hạ `min_support` → số itemset tăng (đúng tính chất anti-monotone); nâng `min_confidence` → số luật giảm.

In [8]:
rows = []
for minsup in [0.2, 0.4, 0.6, 0.8]:
    f = apriori(df, min_support=minsup, use_colnames=True)
    for minconf in [0.6, 0.8, 1.0]:
        r = association_rules(f, metric="confidence", min_threshold=minconf) if len(f) else []
        rows.append({"min_support": minsup, "min_confidence": minconf,
                     "n_itemsets": len(f), "n_rules": len(r)})

pd.DataFrame(rows).pivot(index="min_support", columns="min_confidence", values="n_rules")

min_confidence,0.6,0.8,1.0
min_support,,,
0.2,72,45,45
0.4,40,13,13
0.6,16,6,6
0.8,0,0,0


## 6. Kết luận

- Itemset lớn nhất thoả `minsup = 0.4` là `{Bread, Coke, Diaper, Milk}` (support 0.4 — xuất hiện ở 2/5 giao dịch).
- Các luật mạnh nhất có `confidence = 1.0` và `lift = 1.67`, ví dụ `{Milk, Diaper} => {Coke}`: mọi giao dịch mua Milk và Diaper đều có mua Coke.
- `{Beer} => {Diaper}` cũng có confidence 1.0 nhưng support chỉ 0.4 nên độ phủ thấp.